# Old vs new column rule in `calculate_indices_mapping`: a toy example

`InteractionFingerprint.calculate_indices_mapping` (in `plipify/fingerprints.py`) goes through the
sequence alignment one column at a time and decides:

1. **Keep or drop the column.** A dropped column disappears from the fingerprint for *every* structure.
2. **For a kept column, which residue each structure contributes**, or `None`, which counts as 0.

This notebook runs the **old** and **new** versions of that loop side by side on a small hand-made
alignment, so you can see exactly where they differ. No PDB files or PLIP are needed. MUSCLE is used
only in the last section, to check that the real plipify function gives the same answer.

| | rule | code |
|---|---|---|
| **old** | keep a column only if *every* structure with a residue there has the **same** amino acid | `if len(residue_types) != 1: continue` |
| **new** | keep a column if the most common amino acid covers at least `min_consensus` of the structures; structures with a different amino acid get `None` for that column | `if n_consensus / total < min_consensus: continue` |


## 1. The toy dataset

20 structures, 7 positions, named after Mpro residues so the numbers mean something:

| structures | what they mimic |
|---|---|
| `wt01`–`wt18` | ordinary SARS-CoV-2 Mpro chains |
| `wt02` | same, but its first residue is not resolved (a **gap**, `-`) |
| `homolog` | an Mpro from a related virus (like z5gwy): differs at **M49→L, S144→T, Q189→P** |
| `H41A` | the His41→Ala mutant (like z2q6g): differs at **H41→A** |

The last position, `loop`, is a variable surface loop: 13 structures have K and 7 have R.
This is real disagreement, and both rules should drop it.

The sequences are already aligned (all the same length), so we can skip MUSCLE here. In plipify,
these strings are what MUSCLE returns (`rows`).

In [1]:
from collections import Counter
import pandas as pd

positions = ["H41", "M49", "N142", "S144", "E166", "Q189", "loop"]
wt = "HMNSEQK"

structures = {}
for i in range(1, 19):
    structures[f"wt{i:02d}"] = wt if i <= 11 else wt[:-1] + "R"  # loop: K in wt01-11, R in wt12-18
structures["wt02"] = "-" + wt[1:]   # first residue not resolved
structures["homolog"] = "HLNTEPK"   # differs at M49, S144, Q189
structures["H41A"] = "AMNSEQK"      # His41 -> Ala

names = list(structures)
rows = list(structures.values())    # what MUSCLE would return: one aligned string per structure

alignment = pd.DataFrame([list(r) for r in rows], index=names, columns=positions)

def highlight_odd(col):
    majority = col[col != "-"].mode()[0]
    return ["background-color:#f4b6b6" if v not in (majority, "-") else
            "background-color:#ddd" if v == "-" else "" for v in col]

alignment.style.apply(highlight_odd)

,H41,M49,N142,S144,E166,Q189,loop
wt01,H,M,N,S,E,Q,K
wt02,-,M,N,S,E,Q,K
wt03,H,M,N,S,E,Q,K
wt04,H,M,N,S,E,Q,K
wt05,H,M,N,S,E,Q,K
wt06,H,M,N,S,E,Q,K
wt07,H,M,N,S,E,Q,K
wt08,H,M,N,S,E,Q,K
wt09,H,M,N,S,E,Q,K
wt10,H,M,N,S,E,Q,K


Red cells differ from the majority residue at that position; grey cells are gaps.
Only **2 of the 20 structures** are odd, plus one gap and the K/R loop.

## 2. Toy interaction counts

How many interactions each structure's residue at each position makes with its ligand.
These numbers play the role of PLIP's output. The homologue's counts are deliberately large (3),
so you can see what happens if they end up in the wrong row.

In [2]:
wt_counts = {1: 1, 2: 1, 3: 1, 4: 1, 5: 2, 6: 1, 7: 0}   # per seq_index (= position here)

contacts = {name: dict(wt_counts) for name in names if name.startswith("wt")}
del contacts["wt02"][1]                                   # unresolved residue makes no contacts
contacts["homolog"] = {1: 0, 2: 3, 3: 0, 4: 3, 5: 2, 6: 3, 7: 0}   # L49, T144, P189 interact a lot
contacts["H41A"] = dict(wt_counts)                         # Ala41 still makes 1 (hydrophobic) contact

table = pd.DataFrame(contacts).T.reindex(columns=range(1, 8)).set_axis(positions, axis=1)
table.astype("Int64").astype(object).where(table.notna(), "-")

,H41,M49,N142,S144,E166,Q189,loop
wt01,1,1,1,1,2,1,0
wt02,-,1,1,1,2,1,0
wt03,1,1,1,1,2,1,0
wt04,1,1,1,1,2,1,0
wt05,1,1,1,1,2,1,0
wt06,1,1,1,1,2,1,0
wt07,1,1,1,1,2,1,0
wt08,1,1,1,1,2,1,0
wt09,1,1,1,1,2,1,0
wt10,1,1,1,1,2,1,0


## 3. Shared setup: `columns` (residue number per column)

This is copied from plipify. For each structure it lists, per alignment column, which residue number
(`seq_index`) sits there, or `None` for a gap. Old and new code both use it unchanged.

In [3]:
seq_indices = [[i + 1 for i, c in enumerate(s) if c != "-"] for s in rows]
columns = []
for row, indices in zip(rows, seq_indices):
    residues = iter(indices)
    columns.append([None if c == "-" else next(residues) for c in row])

pd.DataFrame(columns, index=names, columns=positions, dtype=object).loc[["wt01", "wt02", "homolog", "H41A"]]

,H41,M49,N142,S144,E166,Q189,loop
wt01,1,2,3,4,5,6,7
wt02,None,2,3,4,5,6,7
homolog,1,2,3,4,5,6,7
H41A,1,2,3,4,5,6,7


## 4. The two loops, verbatim

Both functions contain the loop body exactly as it is in `fingerprints.py`, old (before the patch)
and new (after). Only the function wrapper is added.

In [4]:
def old_mapping(rows, columns):
    old2new = [{} for _ in rows]
    for col in range(len(rows[0])):                              # aligned.get_alignment_length()
        residue_types = {row[col] for row in rows} - {"-"}       # SET: which letters, not how many
        if len(residue_types) != 1:                              # >1 kind of amino acid ...
            continue                                             # ... drop the column for everyone
        for mapping, structure_columns in zip(old2new, columns):
            mapping[col + 1] = {"seq_index": structure_columns[col], "chain": "any"}
    return old2new


def new_mapping(rows, columns, min_consensus=0.9):
    old2new = [{} for _ in rows]
    for col in range(len(rows[0])):                              # aligned.get_alignment_length()
        residue_types = Counter(row[col] for row in rows if row[col] != "-")   # COUNTER: how many of each
        if not residue_types:
            continue
        consensus, n_consensus = residue_types.most_common(1)[0]
        if n_consensus / sum(residue_types.values()) < min_consensus:          # too little agreement ...
            continue                                                           # ... only then drop
        for mapping, row, structure_columns in zip(old2new, rows, columns):
            # structures with a different residue here count as a gap for this column
            seq_index = structure_columns[col] if row[col] == consensus else None
            mapping[col + 1] = {"seq_index": seq_index, "chain": "any"}
    return old2new


old = old_mapping(rows, columns)
new = new_mapping(rows, columns, min_consensus=0.9)

## 5. Difference 1: which columns survive

For each position: what the old code sees (a **set**), what the new code sees (a **Counter**), the
agreement fraction, and the keep/drop decision of each rule.

In [5]:
summary = []
for col, pos in enumerate(positions):
    letters = [r[col] for r in rows if r[col] != "-"]
    counts = Counter(letters)
    consensus, n = counts.most_common(1)[0]
    summary.append({
        "position": pos,
        "old sees (set)": "{" + ", ".join(sorted(set(letters))) + "}",
        "new sees (Counter)": dict(counts),
        "consensus": consensus,
        "agreement": f"{n}/{len(letters)} = {n / len(letters):.0%}",
        "old keeps?": "yes" if (col + 1) in old[0] else "DROPPED",
        "new keeps?": "yes" if (col + 1) in new[0] else "DROPPED",
    })
summary = pd.DataFrame(summary).set_index("position")

def color_keep(v):
    return "color:#b00020;font-weight:bold" if v == "DROPPED" else "color:#1b7f3b"

summary.style.map(color_keep, subset=["old keeps?", "new keeps?"])

,old sees (set),new sees (Counter),consensus,agreement,old keeps?,new keeps?
position,,,,,,
H41,"{A, H}","{'H': 18, 'A': 1}",H,18/19 = 95%,DROPPED,yes
M49,"{L, M}","{'M': 19, 'L': 1}",M,19/20 = 95%,DROPPED,yes
N142,{N},{'N': 20},N,20/20 = 100%,yes,yes
S144,"{S, T}","{'S': 19, 'T': 1}",S,19/20 = 95%,DROPPED,yes
E166,{E},{'E': 20},E,20/20 = 100%,yes,yes
Q189,"{P, Q}","{'Q': 19, 'P': 1}",Q,19/20 = 95%,DROPPED,yes
loop,"{K, R}","{'K': 13, 'R': 7}",K,13/20 = 65%,DROPPED,DROPPED


**What to notice**

* **Old:** at M49, S144 and Q189, *one* homologue out of 20 makes the set contain two letters, so
  `len(residue_types) != 1` is true and the column is dropped for all 20 structures. The same happens
  at H41 because of the one H41A mutant. Only N142 and E166 survive, **2 of 7 positions**.
* **New:** the Counter shows 19 vs 1, which is 95% agreement, above 0.9, so the column is kept.
  **6 of 7 positions** survive.
* **H41:** the denominator is 19, not 20, because `wt02` has a gap there. Gaps don't count for or
  against agreement in either version.
* **loop:** 13 K vs 7 R is only 65% agreement. The new rule still drops it, so it doesn't keep
  columns where the structures genuinely disagree.

## 6. Difference 2: what each structure contributes to a kept column

Below is the `seq_index` each structure gets per kept column (`None` = contributes 0).
Only the interesting structures are shown; every wt row looks like `wt01`.

In [6]:
def mapping_table(mappings, show=("wt01", "wt02", "homolog", "H41A")):
    table = {}
    for name, m in zip(names, mappings):
        if name in show:
            row = {}
            for col, pos in enumerate(positions, start=1):
                if col not in m:
                    row[pos] = "dropped"                  # column removed for everyone
                elif m[col]["seq_index"] is None:
                    row[pos] = "None"                     # this structure contributes 0 here
                else:
                    row[pos] = m[col]["seq_index"]
            table[name] = row
    return pd.DataFrame(table).T

def shade(v):
    if v == "dropped":
        return "background-color:#eee;color:#999"
    if v == "None":
        return "background-color:#f4b6b6;font-weight:bold"
    return ""

print("OLD")
display(mapping_table(old).style.map(shade))
print("NEW")
display(mapping_table(new).style.map(shade))

OLD


,H41,M49,N142,S144,E166,Q189,loop
wt01,dropped,dropped,3,dropped,5,dropped,dropped
wt02,dropped,dropped,3,dropped,5,dropped,dropped
homolog,dropped,dropped,3,dropped,5,dropped,dropped
H41A,dropped,dropped,3,dropped,5,dropped,dropped


NEW


,H41,M49,N142,S144,E166,Q189,loop
wt01,1,2,3,4,5,6,dropped
wt02,None,2,3,4,5,6,dropped
homolog,1,None,3,None,5,None,dropped
H41A,None,2,3,4,5,6,dropped


**What to notice**

* In the old table, `None` only ever means **gap** (`wt02` at H41, but H41 is dropped anyway).
  A structure with a different amino acid can't reach this step, because its column was already dropped.
* In the new table, `homolog` gets `None` at M49, S144 and Q189, and `H41A` gets `None` at H41.
  These are masked: they add 0 to that row, and everyone else still counts.

## 7. The resulting fingerprint

Sum the toy interaction counts over all structures, using each mapping. This is what
`_calculate_fingerprint_one_structure` and `_acumulate_fingerprints` do in plipify:
`seq_index=None` makes a `GAP` with 0 interactions.

A third column, **new rule without masking**, shows what happens if the threshold is relaxed but
mismatching structures still contribute their own residue. That was the bug caused by the old loop
left in below the new one.

In [7]:
def fingerprint(mappings):
    totals = {}
    for k in mappings[0]:
        total = 0
        for name, m in zip(names, mappings):
            seq_index = m[k]["seq_index"]
            if seq_index is not None:
                total += contacts[name].get(seq_index, 0)
        totals[positions[k - 1]] = total
    return pd.Series(totals, dtype="Int64")


def new_mapping_without_masking(rows, columns, min_consensus=0.9):
    # relaxed threshold, but every structure contributes its own residue (WRONG)
    old2new = [{} for _ in rows]
    for col in range(len(rows[0])):
        residue_types = Counter(row[col] for row in rows if row[col] != "-")
        consensus, n_consensus = residue_types.most_common(1)[0]
        if n_consensus / sum(residue_types.values()) < min_consensus:
            continue
        for mapping, structure_columns in zip(old2new, columns):
            mapping[col + 1] = {"seq_index": structure_columns[col], "chain": "any"}
    return old2new


fp = pd.DataFrame({
    "old (all must agree)": fingerprint(old),
    "new (consensus + masking)": fingerprint(new),
    "new WITHOUT masking (bug)": fingerprint(new_mapping_without_masking(rows, columns)),
}).reindex(positions)
fp.loc["TOTAL"] = fp.sum()
fp.astype(object).where(fp.notna(), "dropped")

,old (all must agree),new (consensus + masking),new WITHOUT masking (bug)
H41,dropped,17,18
M49,dropped,19,22
N142,19,19,19
S144,dropped,19,22
E166,40,40,40
Q189,dropped,19,22
loop,dropped,dropped,dropped
TOTAL,59,133,143


**How to read it**

* **old:** only N142 (19) and E166 (40) remain, 59 interactions in total. H41, M49, S144 and Q189
  are gone even though 18–19 of 20 structures agree on them. This is the same collapse you saw in
  the real Mpro run (59 → 19 rows, 1658 → 1016 interactions).
* **new:** those positions come back. Each one counts only the structures that actually have the
  consensus residue: S144 = 19 (18 wt + H41A); the homologue's T144 contributes nothing.
* **without masking:** S144, M49 and Q189 jump to 22. That is the homologue's 3 contacts from
  *different* amino acids (T, L, P) added to rows labelled S, M and Q. H41 gets the mutant's Ala
  contact. The rows look fuller but are wrong, which is why the `else None` in the new code matters.

## 8. `min_consensus` controls the behaviour

`min_consensus=1.0` reproduces the old rule exactly. Lowering it keeps more columns.

In [8]:
sweep = {}
for th in (1.0, 0.95, 0.9, 0.65, 0.6):
    m = new_mapping(rows, columns, min_consensus=th)
    sweep[f"min_consensus={th}"] = fingerprint(m)
sweep = pd.DataFrame(sweep).reindex(positions)
sweep.loc["positions kept"] = sweep.notna().sum()
display(sweep.astype(object).where(sweep.notna(), "dropped"))

assert new_mapping(rows, columns, min_consensus=1.0) == old, "1.0 should equal the old rule"
print("min_consensus=1.0 gives exactly the old mapping: OK")

,min_consensus=1.0,min_consensus=0.95,min_consensus=0.9,min_consensus=0.65,min_consensus=0.6
H41,dropped,dropped,17,17,17
M49,dropped,19,19,19,19
N142,19,19,19,19,19
S144,dropped,19,19,19,19
E166,40,40,40,40,40
Q189,dropped,19,19,19,19
loop,dropped,dropped,dropped,0,0
positions kept,2,5,6,7,7


min_consensus=1.0 gives exactly the old mapping: OK


* **0.95 drops H41.** Agreement there is 18/19 = 94.7%, just under 0.95, so a single mutant
  is enough to remove it again. The fraction depends on how many structures you have: one dissenter
  among 20 costs 5%, among 558 it costs 0.2%. Small datasets need a lower threshold.
* **0.65 keeps the `loop` column** (13/20 = 65% K). Only the K structures count there; the total is
  0 because this toy's loop makes no contacts.

Picking the threshold is a judgement call: too low and you start merging genuinely different
residues into one row. On the real 558-structure Mpro set, 0.9 and 0.95 both keep every hotspot.

## 9. Check against the real plipify function

The same toy sequences go through the real `InteractionFingerprint.calculate_indices_mapping`
(with MUSCLE). `FakeStructure` provides just `.sequence()`, the same trick as in
`plipify/tests/test_fingerprints.py`. In plipify, unresolved residues are stripped before MUSCLE
and MUSCLE puts the gap back.

In [9]:
import shutil
from plipify.fingerprints import InteractionFingerprint

class FakeStructure:
    def __init__(self, identifier, sequence):
        self.identifier = identifier
        self._sequence = sequence
    def sequence(self):
        return self._sequence

if shutil.which("muscle") is None:
    print("muscle not installed: skipping this check")
else:
    fakes = [FakeStructure(n, s) for n, s in structures.items()]
    real_strict = InteractionFingerprint.calculate_indices_mapping(fakes, min_consensus=1.0)
    real_consensus = InteractionFingerprint.calculate_indices_mapping(fakes, min_consensus=0.9)
    print()
    print("real plipify, min_consensus=1.0 == toy old_mapping :", real_strict == old)
    print("real plipify, min_consensus=0.9 == toy new_mapping :", real_consensus == new)


real plipify, min_consensus=1.0 == toy old_mapping : True
real plipify, min_consensus=0.9 == toy new_mapping : True



muscle 5.2.linux64 [-]  32.1Gb RAM, 8 cores
Built Dec 16 2025 21:26:16
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

Input: 20 seqs, avg length 7, max 7

00:00 9.9Mb  CPU has 8 cores, running 8 threads
00:00 539Mb   100.0% Calc posteriors
00:00 539Mb   100.0% Consistency (1/2)
00:00 539Mb   100.0% Consistency (2/2)
00:00 539Mb   100.0% UPGMA5           
00:00 539Mb   100.0% Refining

muscle 5.2.linux64 [-]  32.1Gb RAM, 8 cores
Built Dec 16 2025 21:26:16
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

Input: 20 seqs, avg length 7, max 7

00:00 9.9Mb  CPU has 8 cores, running 8 threads
00:00 539Mb   100.0% Calc posteriors
00:00 539Mb   100.0% Consistency (1/2)
00:00 539Mb   100.0% Consistency (2/2)
00:00 539Mb   100.0% UPGMA5           
00:00 539Mb   100.0% Refining


## Summary

| | old | new |
|---|---|---|
| looks at | a **set** of letters (which kinds are present) | a **Counter** (how many of each) |
| drops a column when | more than one kind of amino acid is present | agreement < `min_consensus` |
| one odd structure out of many | deletes the column **for everyone** | is masked (`None`) **for that column only** |
| structure with a different amino acid | never reaches the mapping step | gets `seq_index=None` and contributes 0 |
| toy result | 2 of 7 positions, 59 interactions | 6 of 7 positions, 133 interactions |

The one line that caused the collapse in the old code is `if len(residue_types) != 1: continue`.